# 流水线并行Embedding
在流水线并行（Pipeline Parallelism, PP）中，
- 将输入 Embedding 层放在第一个阶段（First Stage），
- 将输出 Embedding 层（或 LM Head）放在最后一个阶段（Last Stage），
是大型语言模型（LLM）训练框架中最经典且高效的模型切分策略。

###### 这种设计主要基于计算逻辑、显存优化和通信效率三个维度的考量：
1. 符合模型天然的前向计算逻辑

Transformer 架构的数据流是严格单向的：输入文本 -> 词嵌入(Embedding) -> 多层 Transformer Blocks -> 语言模型头(LM Head) -> 输出概率。

- First Stage (起点)：
  - 流水线必须从某个地方开始接收原始的 input_ids（文本的 token 编号）。<font color='red'>第一个 GPU 负责将这些离散的 token 编号通过查表转化为连续的稠密向量（即输入 Embedding），这是整个神经网络计算的起点。</font>
- Last Stage (终点)：
  - 经过几十甚至上百层 Transformer 的深层计算后，最终的隐藏状态（Hidden States）需要被<font color='red'>映射回词表空间，计算出每个 token 的概率分布</font>。这个映射过程由 LM Head（通常是一个巨大的线性层）完成，它必须位于流水线的最末端。

2. 极致的显存与通信优化
在大规模分布式训练中，这种切分方式能显著降低跨 GPU 的通信开销：
- 避免传输原始文本：
  - 如果将 Embedding 层放在后面的阶段，前面的 GPU 就需要把原始的 input_ids（虽然数据量小，但需要额外处理）跨卡传递。直接在 First Stage 完成 Embedding 查找，后续阶段之间只需要传递固定大小的隐藏状态向量（Hidden States），通信模式非常规整且高效。
- 减少中间激活值的传输：
  - Embedding 层和 LM Head 通常包含巨大的参数量（与词表大小成正比，例如 10 万+ 的词表）。将它们分别固定在流水线的两端，可以让中间的所有 GPU 专注于计算密集的 Transformer 模块，只负责传递计算结果（激活值），而不需要搬运庞大的词表参数。

3. 带来的工程挑战与解决方案
这种“首尾分离”的架构虽然高效，但也带来了一个核心的工程难题：输入和输出 Embedding 的权重共享（Weight Tying）。

在绝大多数 LLM（如 GPT、LLaMA）中，为了提升模型效果并节省参数，输入 Embedding 矩阵和输出 LM Head 的权重矩阵是共享的（即 tied）。
- 挑战：在流水线并行中，First Stage 和 Last Stage 往往位于完全不同的物理 GPU 上（例如 8 卡训练时，分别在 GPU 0 和 GPU 7）。如果权重是共享的，那么当模型在 Last Stage 计算出梯度并更新 LM Head 的权重时，First Stage 的输入 Embedding 权重也必须同步更新为完全相同的值。
- 解决方案：这正是你之前看到的代码中构建 _EMBEDDING_GROUP 的原因。
  - <font color='red'>框架会在 Last Stage 计算完梯度后，通过专门的 EMBEDDING_GROUP 通信组，将更新后的权重（或梯度）直接广播（Broadcast）或同步给 First Stage。</font>
  - 这样，即使它们相隔甚远（中间隔着好几个流水阶段的 GPU），也能保证首尾的 Embedding 矩阵在每一步训练后都保持严格一致。

## 总结来说，
- 将 Embedding 放在首尾是顺应模型计算流、最大化计算与通信效率的最佳实践。而分布式训练框架（如 Megatron-LM、DeepSpeed）则通过在后台建立专属的“Embedding 通信组”，巧妙地解决了由此产生的权重同步难题。

# 输入和输出的 Embedding
输入和输出的 Embedding 其实正好构成了一个“翻译”的闭环。我们可以用一个非常直观的“查字典”例子来理解它们：

## 📖 输入 Embedding：从“查字典”到“语义向量”
输入 Embedding 的核心任务，是把计算机原本看不懂的“人类词汇（Token ID）”，翻译成神经网络能理解的“数学向量（隐藏状态）”。
假设我们有一个极简的词表（字典）：
- 0 -> "我"
- 1 -> "爱"
- 2 -> "AI"

假设我们设定的隐藏状态向量维度是 3 维（即每个词用 3 个数字来代表它的语义）。模型内部会有一张巨大的“参数表”（Embedding Matrix），比如长这样：
- 代表“我”的向量是 [0.1, 0.5, -0.2]
- 代表“爱”的向量是 [0.8, -0.3, 0.4]
- 代表“AI”的向量是 [-0.5, 0.9, 0.1]

###### 输入过程（查字典）：
当流水线并行的第一个阶段（First Stage）收到原始输入 ["我", "爱", "AI"] 时，它拿到的其实是对应的 ID [0, 1, 2]。
- <font color='red'>输入 Embedding 层的工作就是拿着这些 ID 去上面的“参数表”里查表，把 [0, 1, 2] 瞬间替换成对应的向量：</font>
  - [[0.1, 0.5, -0.2], [0.8, -0.3, 0.4], [-0.5, 0.9, 0.1]]

<font color='red'>这组向量，就是我们上一轮提到的隐藏状态向量的初始形态</font>。接下来，这组向量就会被送入 Transformer 的各层网络中进行复杂的计算。

## 🗣️ 输出 Embedding (LM Head)：从“语义向量”回到“词汇概率”
输出 Embedding（在大模型中通常叫 LM Head，即语言模型头）的任务正好相反。<font color='red'>它要把经过几十层网络深度计算后、包含了极其丰富语境信息的最终隐藏状态向量，重新“翻译”回人类能看懂的词表概率。</font>

- 假设模型经过层层计算，对于第一个词“我”，最终输出了一个融合了全句语境的隐藏状态向量，比如是 [0.7, 0.2, -0.1]。

###### 输出过程（打分与还原）：
<font color='red'>输出 Embedding 层（LM Head）</font>本质上是一个巨大的线性映射层（矩阵乘法）。它会拿这个最终的隐藏状态向量 [0.7, 0.2, -0.1]，去和词表中所有词的向量做内积（也就是计算相似度/打分）。

- 它和“我”的向量 [0.1, 0.5, -0.2] 算一下，得出一个分数（比如 0.9 分）。
- 它和“爱”的向量 [0.8, -0.3, 0.4] 算一下，得出一个分数（比如 0.1 分）。
- 它和“AI”的向量 [-0.5, 0.9, 0.1] 算一下，得出一个分数（比如 0.05 分）。

<font color='red'>经过 Softmax 归一化后，模型就会得出结论：下一个词是“我”的概率最高。这样，模型就成功地从一堆抽象的数学向量，还原出了人类能看懂的下一个字。</font>

## 💡 为什么它们经常是“同一个人”（权重共享）？
- 在 GPT、LLaMA 等大模型中，输入 Embedding 的“参数表”和输出 Embedding（LM Head）的“参数表”通常是同一张（即我们之前提到的权重共享）。

- 这背后的逻辑非常优雅：<font color='red'>一个词“作为输入时的含义”，和它“作为输出时的概率”，在语义空间里应该是一致的。</font>
  - 强行让输入和输出共用一套向量参数，不仅能节省一半的 Embedding 显存，还能让模型在学习时更好地约束语义空间，提升训练效果。
  - 这也正是为什么在流水线并行中，需要专门建立 _EMBEDDING_GROUP 来确保流水线首尾的这份“字典”始终保持同步的原因。